# Chapter 9 Student Colab Notebook: Linear Programming

**Course:** MATH 3260 - Linear Algebra  
**Sections:** 9.1-9.5  
**Audience:** Student version

This notebook is the complete interactive teaching companion for Chapter 9. It contains detailed theory, full proofs and algorithmic justifications, original worked examples, every intermediate simplex tableau and row operation, geometric interpretations, Python visualizations, section worksheets, and guided practice.

> **How to use this notebook**
>
> The material is intentionally extensive. Select the examples appropriate for each class meeting and retain the remaining material as a reference. All display mathematics uses `$$...$$` for reliable Google Colab rendering. Exact fractions are preserved in simplex calculations so that rounding does not conceal the algebra.

> **Student workflow**
>
> Run each computational cell, annotate the displayed tableaus, and complete every worksheet before consulting a separate solution key. The theory, proofs, and worked examples are intentionally retained in full.

## Chapter map

1. **Section 9.1:** Linear inequalities, systems of inequalities, half-planes, feasible regions, and modeling  
2. **Section 9.2:** Two-variable linear programming, the vertex principle, graphical optimization, multiple optima, and unbounded regions  
3. **Section 9.3:** Standard form, slack variables, basic feasible solutions, simplex tableaus, pivoting, maximization, and exceptional cases  
4. **Section 9.4:** Minimization, primal-dual construction, weak and strong duality, complementary slackness, and economic interpretation  
5. **Section 9.5:** Mixed constraints, surplus and artificial variables, Phase I/Phase II, infeasibility, and transportation-style applications

Each section concludes with a terminology table, a worksheet, and complete instructor solutions.

In [ ]:
# Core imports and exact-arithmetic helpers used throughout Chapter 9
import itertools
from fractions import Fraction
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Math, Markdown

sp.init_printing()

def F(x):
    """Convert a number to an exact Fraction."""
    return x if isinstance(x, Fraction) else Fraction(x)

def frac_text(q):
    q = F(q)
    return str(q.numerator) if q.denominator == 1 else f"{q.numerator}/{q.denominator}"

def tableau_dataframe(tableau, columns, row_labels=None):
    data = [[frac_text(v) for v in row] for row in tableau]
    if row_labels is None:
        row_labels = [f"R{i+1}" for i in range(len(tableau)-1)] + ["objective"]
    return pd.DataFrame(data, columns=columns, index=row_labels)

def display_tableau(tableau, columns, row_labels=None, caption=None):
    if caption:
        display(Markdown(f"**{caption}**"))
    display(tableau_dataframe(tableau, columns, row_labels))

def pivot_tableau(tableau, pivot_row, pivot_col):
    """Perform one exact Gauss-Jordan pivot and return a new tableau."""
    T = [[F(v) for v in row] for row in tableau]
    p = T[pivot_row][pivot_col]
    if p == 0:
        raise ValueError("The pivot entry cannot be zero.")
    T[pivot_row] = [v / p for v in T[pivot_row]]
    for i in range(len(T)):
        if i == pivot_row:
            continue
        factor = T[i][pivot_col]
        if factor != 0:
            T[i] = [T[i][j] - factor*T[pivot_row][j] for j in range(len(T[i]))]
    return T

def simplex_max_le(A, b, c, max_iter=30):
    """Exact simplex history for max c^T x subject to Ax<=b, x>=0, b>=0."""
    A = [[F(v) for v in row] for row in A]
    b = [F(v) for v in b]
    c = [F(v) for v in c]
    m, n = len(A), len(c)
    T = []
    for i in range(m):
        T.append(A[i] + [F(1 if i == j else 0) for j in range(m)] + [b[i]])
    T.append([-v for v in c] + [F(0)]*m + [F(0)])
    basis = [n+i for i in range(m)]
    history = [{"tableau": [row[:] for row in T], "basis": basis[:], "pivot": None}]
    for _ in range(max_iter):
        reduced = T[-1][:-1]
        minimum = min(reduced)
        if minimum >= 0:
            return history, "optimal"
        enter = reduced.index(minimum)
        candidates = []
        for i in range(m):
            if T[i][enter] > 0:
                candidates.append((T[i][-1]/T[i][enter], i))
        if not candidates:
            return history, "unbounded"
        ratio, leave = min(candidates, key=lambda pair: (pair[0], pair[1]))
        T = pivot_tableau(T, leave, enter)
        basis[leave] = enter
        history.append({"tableau": [row[:] for row in T], "basis": basis[:],
                        "pivot": (enter, leave, ratio)})
    return history, "iteration limit"

def feasible_vertices_2d(constraints, tol=1e-9):
    """Vertices for constraints a*x+b*y<=c."""
    vertices = []
    for first, second in itertools.combinations(constraints, 2):
        a,b,c = first
        d,e,f = second
        determinant = a*e-b*d
        if abs(determinant) < tol:
            continue
        x = (c*e-b*f)/determinant
        y = (a*f-c*d)/determinant
        if all(a0*x+b0*y <= c0+tol for a0,b0,c0 in constraints):
            if not any(abs(x-u)<tol and abs(y-v)<tol for u,v in vertices):
                vertices.append((x,y))
    return sorted(vertices)

def plot_feasible_region(constraints, xlim=(0,9), ylim=(0,9), title="Feasible region",
                         objective=None, vertices=None):
    xs = np.linspace(xlim[0], xlim[1], 500)
    ys = np.linspace(ylim[0], ylim[1], 500)
    X,Y = np.meshgrid(xs,ys)
    mask = np.ones_like(X, dtype=bool)
    for a,b,c in constraints:
        mask &= (a*X+b*Y <= c+1e-9)
    fig, ax = plt.subplots(figsize=(7,6))
    ax.contourf(X,Y,mask.astype(float), levels=[0.5,1.5], alpha=0.25)
    for a,b,c in constraints:
        if abs(b)>1e-12:
            ax.plot(xs,(c-a*xs)/b, linewidth=1.2)
        elif abs(a)>1e-12:
            ax.axvline(c/a, linewidth=1.2)
    if vertices:
        vx=[p[0] for p in vertices]; vy=[p[1] for p in vertices]
        ax.scatter(vx,vy,s=55)
        for p in vertices:
            ax.text(p[0]+0.08,p[1]+0.08,f"({p[0]:g}, {p[1]:g})")
    if objective is not None:
        c1,c2,levels = objective
        for L in levels:
            if abs(c2)>1e-12:
                ax.plot(xs,(L-c1*xs)/c2, linestyle='--', linewidth=1)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_title(title); ax.grid(True,alpha=0.3)
    plt.show()

> **Original-example policy**
>
> All numerical examples are original. Themes are used selectively from the San Antonio Spurs, Backstreet Boys, hiking, badminton, driving, travel, and R&B music. Every business, event, route, or production interpretation is a fictional teaching model.

# Section 9.1 - Systems of Linear Inequalities

## Learning objectives

Students should be able to:

- determine whether a point satisfies a linear inequality;
- graph a linear inequality using its boundary line and a test point;
- distinguish solid and dashed boundaries;
- graph a system of inequalities as an intersection of half-planes;
- compute and label the vertices of a feasible region;
- translate verbal resource restrictions into linear inequalities.

Graphical problems in this chapter are carried out in

$$
\mathbb R^2
$$

unless stated otherwise.


### Definition: linear inequality in two variables

> A linear inequality has one of the forms
> $$
> ax+by<c,\qquad ax+by\le c,\qquad ax+by>c,\qquad ax+by\ge c,
> $$
> where $a$ and $b$ are not both zero. A point $(x,y)$ is a solution when substitution makes the inequality true.

## Boundary line and the two half-planes

The equation

$$
ax+by=c
$$

is the **boundary line**. It divides the plane into two open half-planes:

$$
ax+by<c
\qquad\text{and}\qquad
ax+by>c.
$$

For a non-strict inequality, the boundary is included and is drawn as a solid line. For a strict inequality, the boundary is excluded and is drawn as a dashed line.

## Why one test point determines the entire side

Write

$$
L(x,y)=ax+by-c.
$$

The boundary line is $L(x,y)=0$, or

$$
y=\frac{c-ax}{b}
$$

when $b\ne0$. The line divides the plane into two half-planes. Since $L$ is continuous, its sign cannot change without crossing the boundary $L=0$. Therefore one test point identifies which side satisfies the inequality.


## Detailed example: graphing one inequality

Graph

$$
2x-y\le4.
$$

First replace $\le$ by $=$:

$$
2x-y=4
\quad\Longleftrightarrow\quad
 y=2x-4.
$$

Because the inequality includes equality, draw the boundary as a solid line. Test the origin:

$$
2(0)-0=0\le4.
$$

Thus the solution is the side containing the origin. Solving directly for $y$ confirms the direction:

$$
2x-y\le4
\Longleftrightarrow
-y\le4-2x
\Longleftrightarrow
 y\ge2x-4.
$$

The inequality reverses when multiplying by $-1$.

In [ ]:
# Visualize 2x-y <= 4
constraints_single = [(2,-1,4)]
plot_feasible_region(constraints_single, xlim=(-4,5), ylim=(-6,6),
                     title=r"Solution of $2x-y\leq4$")

## Systems as intersections

A system requires every inequality to hold simultaneously. If the individual solution sets are $H_1,H_2,\ldots,H_k$, then the feasible set is

$$
F=H_1\cap H_2\cap\cdots\cap H_k.
$$

Each half-plane is convex, and an intersection of convex sets is convex. Therefore, every feasible region formed by linear inequalities is convex.

## Proof that a feasible region is convex

Take any two feasible points $\mathbf u$ and $\mathbf v$, and let

$$
\mathbf w=t\mathbf u+(1-t)\mathbf v,
\qquad 0\le t\le1.
$$

Suppose one constraint is

$$
\mathbf a^T\mathbf x\le b.
$$

Because $\mathbf u$ and $\mathbf v$ are feasible,

$$
\mathbf a^T\mathbf u\le b,
\qquad
\mathbf a^T\mathbf v\le b.
$$

Then

$$
\begin{aligned}
\mathbf a^T\mathbf w
&=\mathbf a^T\bigl(t\mathbf u+(1-t)\mathbf v\bigr)\\
&=t\mathbf a^T\mathbf u+(1-t)\mathbf a^T\mathbf v\\
&\le tb+(1-t)b=b.
\end{aligned}
$$

Thus $\mathbf w$ satisfies every constraint and is feasible. Hence the entire segment joining two feasible points lies in the feasible region.

## Detailed feasible-region example

Consider

$$
\begin{aligned}
x&\ge0,\\
y&\ge0,\\
x+y&\le8,\\
2x+y&\le12,\\
x+2y&\le14.
\end{aligned}
$$

The vertices are found by intersecting boundary lines and retaining only feasible intersections.

- On the $x$-axis, $2x\le12$, so the farthest feasible point is $(6,0)$.
- Intersect $x+y=8$ and $2x+y=12$:

$$
\begin{aligned}
x+y&=8,\\
2x+y&=12.
\end{aligned}
$$

Subtracting gives $x=4$, then $y=4$.

- Intersect $x+y=8$ and $x+2y=14$. Subtracting gives $y=6$, then $x=2$.
- On the $y$-axis, $2y\le14$, so the farthest feasible point is $(0,7)$.

Therefore the vertices are

$$
(0,0),\ (6,0),\ (4,4),\ (2,6),\ (0,7).
$$

In [ ]:
constraints_91 = [(-1,0,0),(0,-1,0),(1,1,8),(2,1,12),(1,2,14)]
vertices_91 = feasible_vertices_2d(constraints_91)
print("Feasible vertices:", vertices_91)
plot_feasible_region(constraints_91, xlim=(0,8), ylim=(0,8),
                     title="A bounded feasible region", vertices=vertices_91)

## Modeling example: hiking and badminton schedule

A fictional weekly activity plan uses

- $x$: number of long hiking sessions;
- $y$: number of badminton sessions.

Suppose:

- at most eight total sessions are available: $x+y\le8$;
- hiking uses two recovery units and badminton uses one, with at most twelve units: $2x+y\le12$;
- hiking uses one travel block and badminton uses two facility blocks, with at most fourteen blocks: $x+2y\le14$;
- negative sessions are impossible: $x\ge0$, $y\ge0$.

This verbal model produces exactly the feasible region above. Linear programming begins by translating each sentence into one linear inequality.

## Section 9.1 terminology

| Term | Meaning |
|---|---|
| Linear inequality | A comparison involving a linear expression |
| Boundary line | The equation obtained by replacing the inequality symbol with equality |
| Half-plane | One of the two regions determined by a line |
| Feasible point | A point satisfying every constraint |
| Feasible region | The intersection of all constraint regions |
| Active constraint | A constraint satisfied with equality at a point |
| Vertex | A corner point formed by active boundaries |
| Bounded region | A feasible region contained inside a sufficiently large circle |
| Unbounded region | A feasible region extending indefinitely |

## Section 9.1 worksheet

1. Test $(2,1)$, $(0,0)$, and $(-1,-4)$ in $2x-y>1$.
2. Graph $3x+2y\le12$, including intercepts, boundary type, and a test point.
3. Find all vertices of $x,y\ge0$, $x+y\le6$, $2x+y\le8$; identify active constraints at $(2,4)$.
4. Determine whether $x,y\ge0$, $x-y\ge1$ is bounded and give a feasible direction.
5. Model the driving/hiking excursion problem with an additional requirement $y\ge2$.
6. Prove convexity of intersections of half-planes.
7. Explain why one test point identifies a half-plane.

# Section 9.2 - Linear Programming in Two Variables

## Learning objectives

Students should be able to:

- identify the decision variables, objective function, and constraints;
- locate the feasible region and all of its vertices;
- use the vertex method to maximize or minimize a linear objective;
- explain why a vertex contains an optimum when an optimum exists;
- recognize multiple optimal solutions, infeasibility, and unboundedness;
- formulate and solve a two-variable application.

### Definition: linear programming problem

> A linear programming problem optimizes a linear objective
> $$
> z=c_1x+c_2y
> $$
> subject to finitely many linear constraints. The decision variables describe controllable quantities, and the feasible region contains all choices satisfying every constraint.

## Level lines of the objective

The equation

$$
c_1x+c_2y=k
$$

is a level line on which the objective has the constant value $k$. All level lines are parallel. Increasing or decreasing $k$ slides the line without rotating it.

Graphically, optimization moves the level line in the improving direction until it last touches the feasible region.

## The vertex principle

> **Theorem: optimal solution at a vertex**
>
> For a nonempty polygonal feasible region, if a linear objective has a finite maximum or minimum, then at least one optimal solution is a vertex. If two distinct feasible points are optimal, then every point on the segment joining them is also optimal.

## Detailed proof of the vertex principle

Let the objective be

$$
f(\mathbf x)=\mathbf c^T\mathbf x.
$$

Suppose an optimal point $\mathbf x^*$ is not a vertex. Then $\mathbf x^*$ lies in a nontrivial line segment contained in the feasible region, so there are distinct feasible points $\mathbf u$ and $\mathbf v$ and a number $0<t<1$ such that

$$
\mathbf x^*=t\mathbf u+(1-t)\mathbf v.
$$

By linearity,

$$
\begin{aligned}
f(\mathbf x^*)
&=\mathbf c^T\bigl(t\mathbf u+(1-t)\mathbf v\bigr)\\
&=t\mathbf c^T\mathbf u+(1-t)\mathbf c^T\mathbf v\\
&=tf(\mathbf u)+(1-t)f(\mathbf v).
\end{aligned}
$$

For a maximum, neither $f(\mathbf u)$ nor $f(\mathbf v)$ can exceed $f(\mathbf x^*)$. A weighted average of two numbers no larger than $f(\mathbf x^*)$ can equal $f(\mathbf x^*)$ only if both equal $f(\mathbf x^*)$. Thus $\mathbf u$ and $\mathbf v$ are also optimal.

If either endpoint is not a vertex, repeat the argument on a larger face. A polygon has finitely many faces, so this process reaches an optimal vertex. The minimum case follows by applying the maximum argument to $-f$.

## Detailed graphical maximization example

A fictional Spurs fan-night stand prepares

- $x$: premium merchandise bundles;
- $y$: standard merchandise bundles.

The resource constraints are

$$
\begin{aligned}
x+y&\le8,\\
2x+y&\le12,\\
x+2y&\le14,\\
x,y&\ge0.
\end{aligned}
$$

The profit objective is

$$
P=70x+60y.
$$

Using the Section 9.1 vertices:

| Vertex | Profit calculation | Profit |
|---|---:|---:|
| $(0,0)$ | $70(0)+60(0)$ | $0$ |
| $(6,0)$ | $70(6)+60(0)$ | $420$ |
| $(4,4)$ | $70(4)+60(4)$ | $520$ |
| $(2,6)$ | $70(2)+60(6)$ | $500$ |
| $(0,7)$ | $70(0)+60(7)$ | $420$ |

Therefore the maximum profit is

$$
\boxed{P_{\max}=520}
$$

at

$$
\boxed{(x,y)=(4,4)}.
$$

### Geometric interpretation

Changing objective coefficients changes the slope of the objective level lines while the feasible region remains fixed. The last feasible vertex touched by a moving level line is the optimizer.


In [ ]:
profits = [(p,70*p[0]+60*p[1]) for p in vertices_91]
print(profits)
plot_feasible_region(
    constraints_91, xlim=(0,8), ylim=(0,8),
    title="Fictional fan-night bundle optimization",
    vertices=vertices_91,
    objective=(70,60,[240,420,520])
)

## Multiple optimal solutions

If an objective level line is parallel to an active boundary edge, the final level line can touch an entire segment. For example, on the feasible region above, maximize

$$
Q=x+y.
$$

The constraint $x+y\le8$ implies $Q\le8$. Every feasible point on the edge from $(4,4)$ to $(2,6)$ satisfies $x+y=8$, so every point on that segment is optimal.

The objective value is unique, but the optimizer is not.

## Unbounded feasible region versus unbounded objective

An unbounded feasible region does not automatically imply that the objective is unbounded.

- If the improving direction points into an unlimited feasible direction, no finite optimum exists.
- If the objective worsens along every recession direction, a finite optimum may still occur at a vertex.

For example, with $x,y\ge0$ and $x+y\ge4$, minimizing $x+y$ has the finite minimum $4$, even though the feasible region is unbounded. Maximizing $x+y$ over the same region is unbounded.

## Modeling checklist

1. Define each decision variable with units.
2. Write the objective and state maximize or minimize.
3. Translate each resource statement into a constraint.
4. Include nonnegativity restrictions.
5. Draw or compute the feasible region.
6. Find every relevant vertex.
7. Evaluate the objective at each vertex.
8. Interpret the optimizer using the original units.
9. Check every constraint and report slack or surplus.

## Section 9.2 terminology

| Term | Meaning |
|---|---|
| Decision variable | A controllable unknown quantity |
| Objective function | The linear quantity being maximized or minimized |
| Constraint | A restriction on the decision variables |
| Feasible solution | A point satisfying every constraint |
| Optimal solution | A feasible point with the best objective value |
| Level line | A line on which the objective is constant |
| Multiple optima | More than one feasible point has the same optimal value |
| Infeasible problem | No point satisfies every constraint |
| Unbounded objective | The objective can improve without limit |

## Section 9.2 worksheet

For $Z=50x+40y$ with $x,y\ge0$, $x+y\le10$, $2x+y\le16$, $x+3y\le21$:

1. Find all feasible vertices.
2. Determine the maximum and all slacks.
3. Replace the objective by $Q=2x+y$ and identify every optimizer.
4. Solve the fictional event model $R=80x+60y$ with $x+y\le9$, $2x+y\le14$, $x+2y\le16$.
5. Diagnose unboundedness for maximizing $x+y$ subject to $x-y\ge1$, $x,y\ge0$.
6. Prove the vertex principle.

# Section 9.3 - The Simplex Method: Maximization

## Learning objectives

Students should be able to:

- convert a maximization problem with $\le$ constraints to standard form;
- introduce and interpret slack variables;
- identify basic and nonbasic variables;
- construct and read a simplex tableau;
- justify the entering-variable and ratio-test rules;
- perform every Gauss-Jordan pivot exactly;
- recognize optimality, unboundedness, degeneracy, and multiple optima;
- solve a multi-variable maximization problem.

## Standard form and slack variables

A maximization problem is in standard form when

$$
\max z=\mathbf c^T\mathbf x
$$

subject to

$$
A\mathbf x\le\mathbf b,
\qquad
\mathbf x\ge\mathbf0,
\qquad
\mathbf b\ge\mathbf0.
$$

For each constraint

$$
\mathbf a_i^T\mathbf x\le b_i,
$$

introduce a slack variable $s_i\ge0$:

$$
\mathbf a_i^T\mathbf x+s_i=b_i.
$$

The slack measures unused capacity.

## Basic solutions and vertices

With $m$ independent constraint equations, choose $m$ variables as **basic variables** and set the remaining variables to zero. Solve for the basic variables.

- A **basic solution** is obtained from such a choice.
- A **basic feasible solution** has all variables nonnegative.

Under the usual nondegeneracy assumptions, basic feasible solutions correspond to vertices of the feasible region. The simplex method moves from one basic feasible solution to an adjacent one while improving the objective.

## Why the entering variable has the most negative objective-row coefficient

We use the tableau convention

$$
z-c_1x_1-\cdots-c_nx_n=0.
$$

After pivoting to canonical form, the bottom row has the structure

$$
z+r_1x_1+\cdots+r_nx_n=z_0.
$$

Equivalently,

$$
z=z_0-r_1x_1-\cdots-r_nx_n.
$$

If a nonbasic variable $x_j$ has $r_j<0$, then increasing $x_j$ from zero initially changes the objective by

$$
-r_jx_j>0.
$$

Thus a negative coefficient indicates a possible improvement. Choosing the most negative coefficient gives the largest immediate improvement per unit of that variable. Other negative choices are mathematically valid, although the number of pivots may differ.

## Why the ratio test preserves feasibility

Suppose $x_j$ enters. In canonical form, each basic variable has the form

$$
x_{B_i}=b_i-a_{ij}x_j
$$

when other nonbasic variables remain zero.

To preserve $x_{B_i}\ge0$:

- if $a_{ij}>0$, then
  $$
  x_j\le\frac{b_i}{a_{ij}};
  $$
- if $a_{ij}=0$, the row does not restrict $x_j$;
- if $a_{ij}<0$, increasing $x_j$ increases that basic variable and creates no upper bound.

Therefore the largest feasible increase is

$$
\min_{a_{ij}>0}\frac{b_i}{a_{ij}}.
$$

The row attaining this minimum contains the departing variable. At that step, its basic value reaches zero exactly.

## Why a nonnegative bottom row proves optimality

At a feasible canonical tableau, suppose

$$
z+r_1x_1+\cdots+r_nx_n=z_0
$$

with every reduced coefficient $r_j\ge0$. Because every nonbasic variable is constrained to be nonnegative,

$$
r_1x_1+\cdots+r_nx_n\ge0.
$$

Hence

$$
z=z_0-\sum_jr_jx_j\le z_0.
$$

The current basic feasible solution achieves $z_0$, so no feasible solution can do better. This proves the simplex optimality test.

## Detailed simplex example

Maximize

$$
z=3x_1+5x_2
$$

subject to

$$
\begin{aligned}
x_1&\le4,\\
x_2&\le6,\\
3x_1+2x_2&\le18,\\
x_1,x_2&\ge0.
\end{aligned}
$$

Add slack variables:

$$
\begin{aligned}
x_1+s_1&=4,\\
x_2+s_2&=6,\\
3x_1+2x_2+s_3&=18,\\
z-3x_1-5x_2&=0.
\end{aligned}
$$

The initial basic feasible solution is

$$
(x_1,x_2,s_1,s_2,s_3)=(0,0,4,6,18),
\qquad z=0.
$$

### Geometry of a simplex pivot

A basic feasible solution corresponds to a vertex. A pivot changes the basis and typically moves to an adjacent vertex. The entering variable selects an improving direction, and the ratio test identifies the first constraint that becomes tight.


In [ ]:
A_93 = [[1,0],[0,1],[3,2]]
b_93 = [4,6,18]
c_93 = [3,5]
history_93, status_93 = simplex_max_le(A_93,b_93,c_93)
columns_93 = ["x1","x2","s1","s2","s3","b"]
for k,item in enumerate(history_93):
    caption = "Initial tableau" if k==0 else f"Tableau after pivot {k}"
    display_tableau(item["tableau"], columns_93, caption=caption)
    if item["pivot"]:
        entering, leaving_row, ratio = item["pivot"]
        print("entering column:", columns_93[entering],
              "departing row:", leaving_row+1,
              "minimum ratio:", frac_text(ratio))
print("status:", status_93)

## Pivot 1 in full detail

The most negative bottom-row entry is $-5$, so $x_2$ enters. The positive entries in the $x_2$ column give ratios

$$
\frac61=6,
\qquad
\frac{18}{2}=9.
$$

The minimum is $6$, so $s_2$ departs. The pivot is already $1$. Apply

$$
R_3\leftarrow R_3-2R_2,
\qquad
R_z\leftarrow R_z+5R_2.
$$

The new tableau represents

$$
(x_1,x_2,s_1,s_2,s_3)=(0,6,4,0,6),
\qquad z=30.
$$

## Pivot 2 in full detail

The remaining negative objective-row coefficient is $-3$, so $x_1$ enters. The positive entries in the $x_1$ column give

$$
\frac41=4,
\qquad
\frac63=2.
$$

Thus $s_3$ departs and the pivot is $3$. First normalize:

$$
R_3\leftarrow\frac13R_3.
$$

Then eliminate the other entries in the $x_1$ column:

$$
R_1\leftarrow R_1-R_3,
\qquad
R_z\leftarrow R_z+3R_3.
$$

The final bottom row is nonnegative. Setting nonbasic variables $s_2=s_3=0$ gives

$$
x_1=2,
\qquad
x_2=6,
\qquad
s_1=2.
$$

Therefore

$$
\boxed{z_{\max}=3(2)+5(6)=36}.
$$

## Interpreting the final solution

At $(x_1,x_2)=(2,6)$:

$$
x_1=2<4,
$$

so the first constraint has slack $s_1=2$.

The other two constraints are active:

$$
x_2=6,
$$

and

$$
3x_1+2x_2=3(2)+2(6)=18.
$$

A positive slack means unused capacity. A zero slack means the corresponding resource is fully used.

## Three-decision-variable extension

Maximize

$$
z=4x_1+3x_2+2x_3
$$

subject to

$$
\begin{aligned}
x_1+x_2+x_3&\le4,\\
2x_1+x_2&\le5,\\
x_2+2x_3&\le6,\\
x_1,x_2,x_3&\ge0.
\end{aligned}
$$

The simplex method is unchanged. The tableau simply has one additional decision-variable column.

In [ ]:
history_3var, status_3var = simplex_max_le(
    [[1,1,1],[2,1,0],[0,1,2]],
    [4,5,6],
    [4,3,2]
)
columns_3var=["x1","x2","x3","s1","s2","s3","b"]
for k,item in enumerate(history_3var):
    display_tableau(item["tableau"],columns_3var,caption=f"Three-variable tableau {k}")
print("status:",status_3var)

The final basic solution is

$$
x_1=\frac52,
\qquad
x_2=0,
\qquad
x_3=\frac32,
$$

with

$$
z=4\left(\frac52\right)+2\left(\frac32\right)=13.
$$

The fractions are exact; no rounding is necessary.

## Exceptional simplex outcomes

### Unbounded objective
If an entering column has no positive constraint entry, the entering variable can increase indefinitely without forcing a basic variable below zero. The maximization problem is unbounded.

### Degeneracy
If the minimum ratio is zero, the next basic feasible solution has a basic variable equal to zero. The objective may fail to improve during that pivot. Degeneracy can cause repeated bases; Bland's rule prevents cycling by choosing the smallest-index eligible variables.

### Multiple optima
If a nonbasic decision variable has zero reduced cost in a final tableau, it can enter without changing the objective. This indicates an alternate optimal basic feasible solution and usually an entire optimal edge.

### Infeasibility
If no initial basic feasible solution exists, Phase I with artificial variables is required. A positive Phase I optimum proves that the original constraints are infeasible.

## Section 9.3 terminology

| Term | Meaning |
|---|---|
| Standard form | Maximization with $A\mathbf x\le\mathbf b$, $\mathbf x\ge0$, $\mathbf b\ge0$ |
| Slack variable | Unused amount in a $\le$ constraint |
| Basic variable | A variable represented by a pivot column |
| Nonbasic variable | A variable set to zero at a basic solution |
| Basic feasible solution | A nonnegative basic solution |
| Entering variable | A nonbasic variable selected to improve the objective |
| Departing variable | The basic variable forced to zero by the ratio test |
| Pivot | The entry used for a Gauss-Jordan basis change |
| Reduced cost | Objective-row coefficient after canonicalization |
| Degeneracy | A basic feasible solution with a zero basic variable |

## Section 9.3 worksheet

Use exact simplex arithmetic for $\max z=3x_1+2x_2$ subject to $x_1\le4$, $x_2\le5$, $x_1+x_2\le6$, $x_1,x_2\ge0$.

1. Form the equations and initial tableau.
2. Show every ratio, pivot, and row operation.
3. Report the optimizer, objective, and slacks.
4. Prove optimality from the final row.
5. Explain zero reduced cost, an entering column with no positive entries, and degeneracy.

# Section 9.4 - The Simplex Method: Minimization and Duality

## Learning objectives

Students should be able to:

- write a minimization problem in standard form;
- construct its dual maximization problem;
- prove and apply weak duality;
- understand the simplex-basis proof of strong duality;
- solve a minimization problem by solving its dual;
- use complementary slackness to connect primal and dual optimizers;
- interpret dual variables as marginal resource values.

## Standard primal minimization and dual maximization

Consider the minimization problem

$$
\begin{aligned}
\min\quad &w=\mathbf c^T\mathbf x\\
\text{subject to}\quad&A\mathbf x\ge\mathbf b,\\
&\mathbf x\ge\mathbf0.
\end{aligned}
$$

Its dual is

$$
\begin{aligned}
\max\quad &z=\mathbf b^T\mathbf y\\
\text{subject to}\quad&A^T\mathbf y\le\mathbf c,\\
&\mathbf y\ge\mathbf0.
\end{aligned}
$$

Rows of the primal become variables of the dual, columns of the primal become constraints of the dual, and the coefficient matrix is transposed.

## Weak duality theorem

> **Weak duality**
>
> For every primal-feasible $\mathbf x$ and dual-feasible $\mathbf y$,
> $$
> \mathbf b^T\mathbf y\le\mathbf c^T\mathbf x.
> $$
> Thus every dual value is a lower bound for every primal minimization value.

## Complete proof of weak duality

Primal feasibility gives

$$
A\mathbf x\ge\mathbf b.
$$

Because $\mathbf y\ge\mathbf0$, multiplying componentwise and summing preserves the inequality:

$$
\mathbf y^TA\mathbf x\ge\mathbf y^T\mathbf b=\mathbf b^T\mathbf y.
$$

Dual feasibility gives

$$
A^T\mathbf y\le\mathbf c.
$$

Because $\mathbf x\ge\mathbf0$,

$$
\mathbf x^TA^T\mathbf y\le\mathbf x^T\mathbf c=\mathbf c^T\mathbf x.
$$

The middle scalar is the same in both expressions:

$$
\mathbf y^TA\mathbf x=\mathbf x^TA^T\mathbf y.
$$

Therefore

$$
\boxed{\mathbf b^T\mathbf y\le\mathbf c^T\mathbf x}.
$$

## Strong duality through an optimal simplex basis

For a primal maximization problem

$$
\max\ \mathbf c^T\mathbf x
\quad\text{subject to}\quad
A\mathbf x\le\mathbf b,
\quad \mathbf x\ge0,
$$

suppose an optimal basis matrix $B$ has been found. The basic solution is

$$
\mathbf x_B=B^{-1}\mathbf b.
$$

Define

$$
\mathbf y^T=\mathbf c_B^TB^{-1}.
$$

At an optimal tableau, every reduced cost satisfies

$$
\mathbf c_N^T-\mathbf y^TN\le0,
$$

so

$$
A^T\mathbf y\ge\mathbf c.
$$

The slack-variable columns are identity columns. Their reduced costs are $-y_i\le0$, so $\mathbf y\ge0$. Thus $\mathbf y$ is dual feasible. Finally,

$$
\mathbf b^T\mathbf y
=\mathbf c_B^TB^{-1}\mathbf b
=\mathbf c_B^T\mathbf x_B.
$$

The primal and dual feasible values are equal. Weak duality says neither can be improved, so both are optimal. This is the simplex-basis proof of strong duality.

## Detailed minimization example

Minimize

$$
w=6x_1+4x_2
$$

subject to

$$
\begin{aligned}
x_1+x_2&\ge5,\\
2x_1+x_2&\ge8,\\
x_1,x_2&\ge0.
\end{aligned}
$$

The dual is

$$
\max z=5y_1+8y_2
$$

subject to

$$
\begin{aligned}
y_1+2y_2&\le6,\\
y_1+y_2&\le4,\\
y_1,y_2&\ge0.
\end{aligned}
$$

In [ ]:
history_dual, status_dual = simplex_max_le(
    [[1,2],[1,1]],
    [6,4],
    [5,8]
)
columns_dual=["y1","y2","s1","s2","b"]
for k,item in enumerate(history_dual):
    display_tableau(item["tableau"],columns_dual,caption=f"Dual tableau {k}")
print("status:",status_dual)

## Dual pivot calculations

Initially $y_2$ enters because $-8$ is most negative. Ratios are

$$
\frac62=3,
\qquad
\frac41=4.
$$

Thus $s_1$ departs. After dividing the first row by $2$ and eliminating the $y_2$ column, $y_1$ still has a negative reduced cost. The next ratios are

$$
\frac{3}{1/2}=6,
\qquad
\frac{1}{1/2}=2.
$$

Thus $s_2$ departs. The final solution is

$$
y_1=2,
\qquad
y_2=2,
\qquad z=26.
$$

By strong duality,

$$
\boxed{w_{\min}=26}.
$$

The coefficients under the dual slack columns in the final objective row recover the primal solution:

$$
\boxed{x_1=3,\qquad x_2=2}.
$$

Check:

$$
6(3)+4(2)=26.
$$

## Complementary slackness

For the primal minimization and dual maximization pair,

$$
x_j\left(c_j-(A^T\mathbf y)_j\right)=0
$$

for every primal variable, and

$$
y_i\left((A\mathbf x)_i-b_i\right)=0
$$

for every primal constraint.

### Why it follows
Weak duality can be written as

$$
\mathbf c^T\mathbf x-\mathbf b^T\mathbf y
=\mathbf x^T(\mathbf c-A^T\mathbf y)
+\mathbf y^T(A\mathbf x-\mathbf b).
$$

Every component in both dot products is nonnegative for feasible $\mathbf x$ and $\mathbf y$. At optimality, strong duality makes the left side zero. A sum of nonnegative terms can be zero only when every term is zero. This yields the complementary-slackness equations.

## Complementary-slackness check for the example

The primal solution is $\mathbf x=(3,2)$ and the dual solution is $\mathbf y=(2,2)$.

Both primal constraints are active:

$$
3+2=5,
$$

$$
2(3)+2=8.
$$

Both dual constraints are also active:

$$
2+2(2)=6,
$$

$$
2+2=4.
$$

Because all four variables are positive, complementary slackness correctly requires all four associated constraints to be active.

## Economic interpretation of dual variables

A dual variable is often called a **shadow price**. It estimates how the optimal objective changes when the corresponding right-hand side changes by one unit while the current optimal basis remains valid.

In the example, $y_1=2$ and $y_2=2$. Increasing either minimum requirement by one unit raises the minimum cost by approximately two objective units within the allowable sensitivity range.

This interpretation is local: a sufficiently large change can alter the optimal basis and hence the shadow prices.

## Section 9.4 terminology

| Term | Meaning |
|---|---|
| Primal problem | The original optimization problem |
| Dual problem | The transposed companion optimization problem |
| Weak duality | Every dual feasible value bounds every primal feasible value |
| Strong duality | Finite primal and dual optima have equal values |
| Dual variable | A variable associated with one primal constraint |
| Shadow price | Marginal value of relaxing or tightening a constraint |
| Complementary slackness | Product conditions linking positive variables and active constraints |
| Duality gap | Primal value minus dual value; zero at optimality |

## Section 9.4 worksheet

For $\min w=5x_1+4x_2$ subject to $x_1+x_2\ge5$, $2x_1+x_2\ge8$:

1. Construct and solve the dual with all tableaus.
2. Recover and verify the primal optimizer.
3. Check complementary slackness.
4. Prove weak duality and explain equality as an optimality certificate.
5. Construct the dual of $\min 7x_1+5x_2$ with $x_1+x_2\ge6$, $3x_1+x_2\ge10$.
6. Interpret shadow prices.

# Section 9.5 - Mixed Constraints and the Two-Phase Simplex Method

## Learning objectives

Students should be able to:

- distinguish slack, surplus, and artificial variables;
- explain why a $\ge$ constraint does not provide an obvious basic variable;
- formulate and solve the Phase I auxiliary problem;
- prove the Phase I feasibility criterion;
- remove artificial variables and restore the original objective in Phase II;
- convert mixed minimization to maximization when appropriate;
- recognize infeasibility from a positive Phase I optimum;
- formulate transportation and allocation applications.

## Slack, surplus, and artificial variables

For

$$
\mathbf a^T\mathbf x\le b,
$$

add a slack variable:

$$
\mathbf a^T\mathbf x+s=b,
\qquad s\ge0.
$$

For

$$
\mathbf a^T\mathbf x\ge b,
$$

subtract a surplus variable:

$$
\mathbf a^T\mathbf x-s=b,
\qquad s\ge0.
$$

The coefficient of $s$ is $-1$, so it does not form an identity column for an initial basis. Add an artificial variable:

$$
\mathbf a^T\mathbf x-s+a=b,
\qquad a\ge0.
$$

The artificial variable is temporary and must equal zero before solving the original problem.

## Phase I and Phase II

### Phase I
Minimize the sum of all artificial variables:

$$
w=a_1+\cdots+a_k.
$$

Equivalently, maximize $-w$. Pivot until the auxiliary objective is optimal.

### Phase II
If the Phase I minimum is zero:

1. remove all artificial-variable columns;
2. remove any redundant zero rows;
3. restore the original objective;
4. eliminate coefficients of current basic variables from the objective row;
5. continue ordinary simplex pivots.

## Proof of the Phase I feasibility criterion

Every artificial variable satisfies $a_i\ge0$, so

$$
w=a_1+\cdots+a_k\ge0.
$$

- If Phase I attains $w=0$, then every $a_i=0$. The remaining original, slack, and surplus variables satisfy the original constraints, so the original problem is feasible.
- Conversely, if the original problem has a feasible solution, set every artificial variable to zero. This gives a feasible Phase I solution with $w=0$. Therefore the Phase I minimum must be zero.
- If the Phase I minimum is positive, no feasible point can make all artificial variables zero. Hence the original constraints are infeasible.

Thus

$$
\boxed{\text{original problem feasible}\iff w_{\min}=0}.
$$

## Detailed two-phase example

Maximize

$$
z=5x_1+4x_2
$$

subject to

$$
\begin{aligned}
x_1+x_2&\le6,\\
2x_1+x_2&\ge4,\\
x_1,x_2&\ge0.
\end{aligned}
$$

Introduce variables:

$$
\begin{aligned}
x_1+x_2+s_1&=6,\\
2x_1+x_2-s_2+a_2&=4.
\end{aligned}
$$

For Phase I, maximize $-w=-a_2$. Because $a_2$ is initially basic, eliminate it from the objective row by subtracting the second constraint row.

In [ ]:
# Columns: x1, x2, s1, s2, a2, b
T_phase1_0 = [
    [F(1),F(1),F(1),F(0),F(0),F(6)],
    [F(2),F(1),F(0),F(-1),F(1),F(4)],
    [F(-2),F(-1),F(0),F(1),F(0),F(-4)]
]
cols_phase1=["x1","x2","s1","s2","a2","b"]
display_tableau(T_phase1_0,cols_phase1,caption="Canonical Phase I tableau")
T_phase1_1=pivot_tableau(T_phase1_0,1,0)
display_tableau(T_phase1_1,cols_phase1,caption="After x1 enters and a2 departs")

## Phase I pivot in detail

The Phase I bottom row has $-2$ under $x_1$ and $-1$ under $x_2$, so choose $x_1$ to enter. Ratios are

$$
\frac61=6,
\qquad
\frac42=2.
$$

Thus the artificial variable $a_2$ departs. Normalize and eliminate:

$$
R_2\leftarrow\frac12R_2,
$$

$$
R_1\leftarrow R_1-R_2,
\qquad
R_w\leftarrow R_w+2R_2.
$$

The Phase I objective value is zero and $a_2$ is nonbasic. The original problem is feasible.

In [ ]:
# Remove artificial column and restore z-5x1-4x2=0.
T_phase2_0 = [
    [F(0),Fraction(1,2),F(1),Fraction(1,2),F(4)],
    [F(1),Fraction(1,2),F(0),Fraction(-1,2),F(2)],
    [F(0),Fraction(-3,2),F(0),Fraction(-5,2),F(10)]
]
cols_phase2=["x1","x2","s1","s2","b"]
display_tableau(T_phase2_0,cols_phase2,caption="Initial Phase II tableau")
T_phase2_1=pivot_tableau(T_phase2_0,1,1)
display_tableau(T_phase2_1,cols_phase2,caption="Phase II: x2 enters")
T_phase2_2=pivot_tableau(T_phase2_1,0,3)
display_tableau(T_phase2_2,cols_phase2,caption="Phase II: surplus variable s2 enters")
T_phase2_3=pivot_tableau(T_phase2_2,1,0)
display_tableau(T_phase2_3,cols_phase2,caption="Final Phase II tableau")

## Phase II pivots in detail

After restoring the original objective and eliminating the basic $x_1$ coefficient, the tableau is

$$
\begin{array}{c|rrrr|r}
&x_1&x_2&s_1&s_2&b\\\hline
R_1&0&\frac12&1&\frac12&4\\
R_2&1&\frac12&0&-\frac12&2\\\hline
R_z&0&-\frac32&0&-\frac52&10.
\end{array}
$$

Any negative reduced cost can be selected. To illustrate this flexibility, we choose $x_2$ even though $s_2$ has the most negative coefficient.

### Pivot 1: $x_2$ enters

The eligible ratios are

$$
\frac{4}{1/2}=8,
\qquad
\frac{2}{1/2}=4.
$$

Thus $x_1$ leaves. Apply

$$
R_2\leftarrow2R_2,
\qquad
R_1\leftarrow R_1-\frac12R_2,
\qquad
R_z\leftarrow R_z+\frac32R_2.
$$

The result is

$$
\begin{array}{c|rrrr|r}
&x_1&x_2&s_1&s_2&b\\\hline
R_1&-1&0&1&1&2\\
R_2&2&1&0&-1&4\\\hline
R_z&3&0&0&-4&16.
\end{array}
$$

### Pivot 2: $s_2$ enters

Only $R_1$ has a positive entry in the $s_2$ column, so $s_1$ leaves. Apply

$$
R_2\leftarrow R_2+R_1,
\qquad
R_z\leftarrow R_z+4R_1.
$$

Then

$$
\begin{array}{c|rrrr|r}
&x_1&x_2&s_1&s_2&b\\\hline
R_1&-1&0&1&1&2\\
R_2&1&1&1&0&6\\\hline
R_z&-1&0&4&0&24.
\end{array}
$$

### Pivot 3: $x_1$ enters

Only $R_2$ has a positive entry in the $x_1$ column, so $x_2$ leaves. Apply

$$
R_1\leftarrow R_1+R_2,
\qquad
R_z\leftarrow R_z+R_2.
$$

The final tableau is

$$
\begin{array}{c|rrrr|r}
&x_1&x_2&s_1&s_2&b\\\hline
R_1&0&1&2&1&8\\
R_2&1&1&1&0&6\\\hline
R_z&0&1&5&0&30.
\end{array}
$$

The nonbasic variables are $x_2=s_1=0$, so

$$
x_1=6,
\qquad
s_2=8,
\qquad
\boxed{z_{\max}=30}.
$$

The final row gives

$$
z+x_2+5s_1=30,
$$

which proves $z\le30$ for every feasible solution.

## Mixed minimization by objective negation

To minimize

$$
w=\mathbf c^T\mathbf x,
$$

one may maximize

$$
z=-\mathbf c^T\mathbf x.
$$

The feasible set is unchanged, and

$$
w_{\min}=-z_{\max}.
$$

Mixed $\ge$, $\le$, and equality constraints still require the same surplus/artificial-variable treatment. Objective negation changes only the objective row, not the constraint equations.

## Transportation-style application

A fictional travel-equipment distributor ships equipment from two depots to two hiking centers. After using demand equations to eliminate two shipment variables, the model can produce both capacity constraints of type $\le$ and demand constraints of type $\ge$.

A complete model should include:

- supply limits at each depot;
- minimum demand at each center;
- nonnegative shipment quantities;
- a linear shipping-cost objective.

Transportation problems have additional structure and can be solved by specialized algorithms, but the two-phase simplex method remains valid.

A concrete two-depot/two-center model uses shipment variables $x_{ij}\ge0$ with supply constraints

$$
x_{11}+x_{12}\le S_1,\qquad x_{21}+x_{22}\le S_2,
$$

demand requirements

$$
x_{11}+x_{21}\ge D_1,\qquad x_{12}+x_{22}\ge D_2,
$$

and linear shipping cost

$$
\min\;c_{11}x_{11}+c_{12}x_{12}+c_{21}x_{21}+c_{22}x_{22}.
$$


## Section 9.5 terminology

| Term | Meaning |
|---|---|
| Surplus variable | Amount by which a $\ge$ requirement is exceeded |
| Artificial variable | Temporary variable used to create an initial basis |
| Phase I | Auxiliary optimization that searches for feasibility |
| Phase II | Optimization of the original objective after feasibility is found |
| Big-M method | Alternative method assigning a large penalty to artificial variables |
| Mixed constraints | A model containing different inequality directions or equalities |
| Infeasible tableau | Phase I ends with positive artificial-variable sum |
| Redundant constraint | A constraint that does not change the feasible set |

The **Big-M method** is an alternative to two-phase simplex. Artificial variables are kept in the original objective but assigned a very large penalty $M$. This course emphasizes two-phase simplex because it separates feasibility from optimization more cleanly.


## Section 9.5 worksheet

Use two-phase simplex for $\max z=4x_1+3x_2$ subject to $x_1+x_2\le5$, $2x_1+x_2\ge4$.

1. Introduce slack, surplus, and artificial variables.
2. Complete Phase I and prove feasibility.
3. Restore the objective and complete Phase II.
4. Verify the optimizer, slack, and surplus.
5. Classify added variables for $\le$, $\ge$, and equality constraints.
6. Prove the Phase I criterion.
7. Convert a mixed minimization objective by negation.
8. Formulate a two-depot/two-center transportation model.

# Chapter 9 synthesis

Linear programming links geometry, algebra, and algorithms:

1. Linear inequalities create convex feasible regions.
2. Linear objectives attain finite optima at vertices.
3. Basic feasible solutions are the algebraic representation of those vertices.
4. A simplex pivot moves from one basis to an adjacent basis.
5. Reduced costs determine whether improvement is possible.
6. Duality gives certificates and bounds for optimality.
7. Complementary slackness identifies which variables and constraints can be simultaneously positive or slack.
8. Artificial variables turn feasibility itself into an optimization problem.

The simplex method is therefore not a collection of arbitrary tableau rules. Every step follows from feasibility, linearity, convexity, and exact row-equivalent reformulation.